# 💰 Extraction Fichier Fiscal — Tosyali Iron Steel Algérie

**Objectif** : Extraire le tableau des salaires depuis le PDF fiscal (texte natif) et enrichir avec les numéros de compte depuis le fichier banque.

**Méthode** : `pdfplumber` (pas besoin de VLM — le PDF contient du texte)

**Matching en 2 passes :**
1. **RechercheV exacte** — suppression de tous les espaces avant comparaison
2. **Fuzzy** — uniquement pour les non trouvés en passe 1

> Réutilisable chaque mois — modifier uniquement la cellule **2. Config**

## 1. Imports

In [ ]:
import re
import unicodedata
from pathlib import Path
from datetime import datetime
from collections import Counter

import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# pdfplumber pour PDF texte natif
try:
    import pdfplumber
    print("✅ pdfplumber disponible")
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "pdfplumber", "--quiet"])
    import pdfplumber
    print("✅ pdfplumber installé")

# rapidfuzz pour matching fuzzy
try:
    from rapidfuzz import process as fuzz_process, fuzz
    print("✅ rapidfuzz disponible")
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "rapidfuzz", "--quiet"])
    from rapidfuzz import process as fuzz_process, fuzz
    print("✅ rapidfuzz installé")

print('✅ Imports OK')

## 2. Config  ⬅️ À MODIFIER CHAQUE MOIS

In [ ]:
# ──────────────────────────────────────────────────────────────
#  ⬇️  MODIFIER ICI CHAQUE MOIS
# ──────────────────────────────────────────────────────────────
PDF_PATH    = "/mnt/data/fiscal_fevrier2026.pdf"      # ← PDF fiscal texte
BANQUE_PATH = "/mnt/data/fichier_banque.xlsx"         # ← fichier banque
PERIODE     = "2026-02"                               # ← période AAAA-MM

# Noms des colonnes dans le fichier banque
COL_NOM_BANQUE    = "NOM_PRENOM"     # ← colonne nom complet
COL_COMPTE_BANQUE = "NUMERO_COMPTE"  # ← colonne numéro de compte
# ──────────────────────────────────────────────────────────────

FUZZY_THRESHOLD = 70   # Score minimum (0-100) pour accepter un match fuzzy

OUTPUT_DIR  = Path("/mnt/data/transferts_out")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

date_str   = datetime.now().strftime("%Y%m%d_%H%M")
EXCEL_PATH = OUTPUT_DIR / f"fiscal_{PERIODE}_{date_str}.xlsx"

print(f"PDF fiscal      : {PDF_PATH}")
print(f"Fichier banque  : {BANQUE_PATH}")
print(f"Période         : {PERIODE}")
print(f"Fuzzy threshold : {FUZZY_THRESHOLD}%")
print(f"Excel output    : {EXCEL_PATH}")

## 3. Extraction PDF fiscal (texte natif)

In [ ]:
def extraire_tableau_fiscal(pdf_path: str) -> list:
    """
    Extrait toutes les lignes du tableau fiscal depuis le PDF texte natif.
    Colonnes attendues :
      SIRA | MATRICULE | Nom et Prénom | Adresse |
      SALAIRE BRUTE | IRS | SALAIRE NET | MONTANT TRANSFERABLE | BANQUE
    Retourne une liste de dicts par employé.
    """
    lignes = []

    with pdfplumber.open(pdf_path) as pdf:
        total_pages = len(pdf.pages)
        print(f"   PDF ouvert : {total_pages} pages")

        for page_num, page in enumerate(pdf.pages, 1):
            # Extraction tableau avec pdfplumber
            tables = page.extract_tables()

            if not tables:
                print(f"   Page {page_num} : aucun tableau détecté")
                continue

            for table in tables:
                for row in table:
                    if not row or all(c is None for c in row):
                        continue

                    # Nettoyer les cellules
                    row_clean = [str(c).strip().replace('\n', ' ') if c else '' for c in row]

                    # Ignorer les lignes d'en-tête
                    if any(kw in ' '.join(row_clean).upper() for kw in
                           ['SIRA', 'MATRICULE', 'NOM ET PRENOM', 'SALAIRE', 'TRANSFERABLE', 'BANQUE']):
                        continue

                    # Ignorer les lignes vides ou trop courtes
                    if len(row_clean) < 7:
                        continue

                    # Ignorer si pas de matricule (ligne parasite)
                    if not row_clean[1] or not row_clean[1].isdigit():
                        continue

                    def parse_montant(val: str) -> float:
                        """Convertit '1 441 773,48' ou '1441773.48' en float."""
                        if not val:
                            return None
                        val = re.sub(r'[\s\xa0]', '', val)  # espaces insécables
                        val = val.replace(',', '.')
                        try:
                            return float(val)
                        except:
                            return None

                    # Mapping colonnes :
                    # [0]=SIRA [1]=MATRICULE [2]=NOM_PRENOM [3]=ADRESSE
                    # [4]=SAL_BRUT [5]=IRG [6]=SAL_NET [7]=MONT_TRANSF [8]=BANQUE
                    ligne = {
                        "sira"               : row_clean[0],
                        "matricule"          : row_clean[1],
                        "nom_prenom"         : row_clean[2],
                        "salaire_brut"       : parse_montant(row_clean[4]) if len(row_clean) > 4 else None,
                        "irg"                : parse_montant(row_clean[5]) if len(row_clean) > 5 else None,
                        "salaire_net"        : parse_montant(row_clean[6]) if len(row_clean) > 6 else None,
                        "montant_transferable": parse_montant(row_clean[7]) if len(row_clean) > 7 else None,
                        "banque"             : row_clean[8] if len(row_clean) > 8 else '',
                        "page_pdf"           : page_num,
                    }
                    lignes.append(ligne)

            print(f"   Page {page_num:>3} : {len([l for l in lignes if l['page_pdf'] == page_num]):>3} lignes extraites")

    return lignes


print("Extraction PDF fiscal...")
all_lignes = extraire_tableau_fiscal(PDF_PATH)

print(f"\n✅ Extraction terminée : {len(all_lignes)} employés")

# Aperçu
print(f"\n{'MATR':<8} {'NOM PRÉNOM':<30} {'SAL BRUT':>14} {'IRG':>12} {'SAL NET':>14} {'TRANSF':>14}")
print("-" * 95)
for l in all_lignes[:5]:
    print(f"{l['matricule']:<8} {str(l['nom_prenom'])[:29]:<30} "
          f"{str(l['salaire_brut'] or ''):>14} {str(l['irg'] or ''):>12} "
          f"{str(l['salaire_net'] or ''):>14} {str(l['montant_transferable'] or ''):>14}")

## 4. Chargement fichier banque + fonctions matching

In [ ]:
def normaliser_exact(nom: str) -> str:
    """Supprime TOUS les espaces + majuscules + sans accents."""
    if not nom:
        return ""
    nom = unicodedata.normalize('NFD', str(nom))
    nom = ''.join(c for c in nom if unicodedata.category(c) != 'Mn')
    return re.sub(r'\s+', '', nom.upper())


def normaliser_fuzzy(nom: str) -> str:
    """Espaces simples + majuscules + sans accents."""
    if not nom:
        return ""
    nom = unicodedata.normalize('NFD', str(nom))
    nom = ''.join(c for c in nom if unicodedata.category(c) != 'Mn')
    return re.sub(r'\s+', ' ', nom.upper().strip())


def charger_banque(path: str, col_nom: str, col_compte: str) -> tuple:
    wb  = openpyxl.load_workbook(path)
    ws  = wb.active
    headers       = [str(c.value).strip() if c.value else "" for c in ws[1]]
    headers_upper = [h.upper() for h in headers]
    try:
        idx_nom    = headers_upper.index(col_nom.upper())
        idx_compte = headers_upper.index(col_compte.upper())
    except ValueError:
        print(f"⚠️  Colonnes disponibles : {headers}")
        raise ValueError(f"Colonne '{col_nom}' ou '{col_compte}' introuvable.")

    banque = {}
    for row in ws.iter_rows(min_row=2, values_only=True):
        nom    = row[idx_nom]
        compte = row[idx_compte]
        if nom and compte:
            cle = normaliser_exact(str(nom))
            banque[cle] = {
                "nom_original" : str(nom).strip(),
                "nom_fuzzy"    : normaliser_fuzzy(str(nom)),
                "numero_compte": str(compte).strip(),
            }
    # Index fuzzy
    banque_fuzzy = {v["nom_fuzzy"]: k for k, v in banque.items()}
    return banque, banque_fuzzy


def trouver_compte(nom: str) -> dict:
    """Passe 1 : exact (sans espaces) → Passe 2 : fuzzy si non trouvé."""
    if not nom:
        return {"numero_compte": None, "match_type": "NON TROUVÉ", "nom_banque": None, "score": None}

    # Passe 1 — RechercheV exacte
    cle = normaliser_exact(nom)
    if cle in banque_dict:
        e = banque_dict[cle]
        return {"numero_compte": e["numero_compte"], "match_type": "EXACT",
                "nom_banque": e["nom_original"], "score": 100}

    # Passe 2 — Fuzzy
    result = fuzz_process.extractOne(
        normaliser_fuzzy(nom),
        list(banque_fuzzy_keys.keys()),
        scorer=fuzz.token_sort_ratio,
        score_cutoff=FUZZY_THRESHOLD,
    )
    if result:
        best_key, score, _ = result
        e = banque_dict[banque_fuzzy_keys[best_key]]
        return {"numero_compte": e["numero_compte"], "match_type": "FUZZY",
                "nom_banque": e["nom_original"], "score": round(score)}

    return {"numero_compte": None, "match_type": "NON TROUVÉ", "nom_banque": None, "score": None}


banque_dict, banque_fuzzy_keys = charger_banque(BANQUE_PATH, COL_NOM_BANQUE, COL_COMPTE_BANQUE)
print(f"✅ Fichier banque : {len(banque_dict)} employés chargés")

## 5. Matching numéros de compte

In [ ]:
print("Matching numéros de compte...")
stats = Counter()

for ligne in all_lignes:
    result = trouver_compte(ligne.get("nom_prenom", ""))
    ligne["numero_compte"] = result["numero_compte"]
    ligne["match_type"]    = result["match_type"]
    ligne["nom_banque"]    = result["nom_banque"]
    ligne["score_match"]   = result["score"]
    stats[result["match_type"]] += 1

print(f"\n📊 Résultats matching :")
print(f"   ✅ EXACT       : {stats['EXACT']:>4} employés")
print(f"   🔶 FUZZY       : {stats['FUZZY']:>4} employés")
print(f"   ❌ NON TROUVÉ  : {stats['NON TROUVÉ']:>4} employés")
print(f"   ──────────────────────────")
print(f"   Total          : {len(all_lignes):>4} employés")

# Détail matchs fuzzy
fuzzy_list = [l for l in all_lignes if l["match_type"] == "FUZZY"]
if fuzzy_list:
    print(f"\n🔶 Matchs FUZZY à vérifier :")
    print(f"   {'NOM FISCAL':<30} {'NOM BANQUE':<30} {'SCORE':>6}")
    print("   " + "-" * 70)
    for l in fuzzy_list:
        print(f"   {str(l.get('nom_prenom',''))[:29]:<30} "
              f"{str(l.get('nom_banque',''))[:29]:<30} "
              f"{str(l.get('score_match','')):>6}%")

# Détail non trouvés
non_trouves = [l for l in all_lignes if l["match_type"] == "NON TROUVÉ"]
if non_trouves:
    print(f"\n❌ NON TROUVÉS dans le fichier banque :")
    for l in non_trouves:
        print(f"   - {l.get('matricule','')} | {l.get('nom_prenom','')}")

## 6. Export Excel

In [ ]:
def export_excel(lignes: list, output_path: Path, periode: str):
    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = f"Fiscal {periode}"

    # ── Styles ──
    thin         = Side(style="thin")
    border       = Border(left=thin, right=thin, top=thin, bottom=thin)
    header_font  = Font(name="Calibri", bold=True, color="FFFFFF", size=10)
    header_fill  = PatternFill("solid", fgColor="1F4E79")
    header_align = Alignment(horizontal="center", vertical="center", wrap_text=True)
    cell_align   = Alignment(horizontal="left",   vertical="center")
    amt_align    = Alignment(horizontal="right",  vertical="center")
    ctr_align    = Alignment(horizontal="center", vertical="center")
    total_fill   = PatternFill("solid", fgColor="BDD7EE")
    exact_fill   = PatternFill("solid", fgColor="E2EFDA")  # vert
    fuzzy_fill   = PatternFill("solid", fgColor="FFF2CC")  # jaune
    notfnd_fill  = PatternFill("solid", fgColor="FCE4D6")  # rouge

    # ── Titre ──
    ws.merge_cells("A1:L1")
    tc = ws["A1"]
    tc.value     = f"Tosyali Iron Steel Algérie — État des Salaires — {periode}"
    tc.font      = Font(name="Calibri", bold=True, size=13, color="1F4E79")
    tc.alignment = Alignment(horizontal="center", vertical="center")
    ws.row_dimensions[1].height = 28

    # ── Sous-titre ──
    ws.merge_cells("A2:L2")
    n_ex = sum(1 for l in lignes if l.get('match_type') == 'EXACT')
    n_fz = sum(1 for l in lignes if l.get('match_type') == 'FUZZY')
    n_nf = sum(1 for l in lignes if l.get('match_type') == 'NON TROUVÉ')
    sc = ws["A2"]
    sc.value = (
        f"Extrait le {datetime.now().strftime('%d/%m/%Y %H:%M')} — {len(lignes)} employés  |  "
        f"✅ Exact: {n_ex}   🔶 Fuzzy: {n_fz}   ❌ Non trouvé: {n_nf}"
    )
    sc.font      = Font(name="Calibri", italic=True, size=10, color="595959")
    sc.alignment = Alignment(horizontal="center")

    # ── Légende ──
    ws.merge_cells("A3:L3")
    leg = ws["A3"]
    leg.value     = "Couleurs : vert = match exact   |   jaune = match fuzzy (à vérifier)   |   rouge = non trouvé"
    leg.font      = Font(name="Calibri", italic=True, size=9, color="595959")
    leg.alignment = Alignment(horizontal="center")

    # ── En-têtes 12 colonnes ──
    headers = [
        "N°", "Matricule", "Nom & Prénom",
        "Salaire Brut", "IRG", "Salaire Net", "Montant Transférable",
        "Banque", "Période",
        "N° Compte", "Nom Banque (match)", "Type Match"
    ]
    col_widths = [5, 10, 30, 16, 14, 16, 20, 20, 10, 22, 30, 12]

    for col, (h, w) in enumerate(zip(headers, col_widths), 1):
        cell           = ws.cell(row=4, column=col, value=h)
        cell.font      = header_font
        cell.fill      = header_fill
        cell.alignment = header_align
        cell.border    = border
        ws.column_dimensions[get_column_letter(col)].width = w
    ws.row_dimensions[4].height = 30

    # ── Données ──
    tot_brut = tot_irg = tot_net = tot_transf = 0.0

    for idx, ligne in enumerate(lignes, 1):
        row = idx + 4
        mt  = ligne.get("match_type", "NON TROUVÉ")
        row_fill = exact_fill if mt == "EXACT" else (fuzzy_fill if mt == "FUZZY" else notfnd_fill)

        s_brut  = ligne.get("salaire_brut")        or 0.0
        s_irg   = ligne.get("irg")                 or 0.0
        s_net   = ligne.get("salaire_net")         or 0.0
        s_transf= ligne.get("montant_transferable") or 0.0
        tot_brut  += s_brut
        tot_irg   += s_irg
        tot_net   += s_net
        tot_transf+= s_transf

        values = [
            idx,
            ligne.get("matricule", ""),
            ligne.get("nom_prenom", ""),
            s_brut   or None,
            s_irg    or None,
            s_net    or None,
            s_transf or None,
            ligne.get("banque", ""),
            periode,
            ligne.get("numero_compte", "") or "",
            ligne.get("nom_banque",    "") or "",
            mt,
        ]
        for col, val in enumerate(values, 1):
            cell        = ws.cell(row=row, column=col, value=val)
            cell.border = border
            cell.fill   = row_fill
            if col in (4, 5, 6, 7) and val is not None:
                cell.number_format = '#,##0.00'
                cell.alignment     = amt_align
            elif col in (1, 2, 12):
                cell.alignment = ctr_align
            else:
                cell.alignment = cell_align

    # ── Ligne TOTAL ──
    total_row = len(lignes) + 5
    ws.merge_cells(f"A{total_row}:C{total_row}")
    tl = ws.cell(row=total_row, column=1, value="TOTAL")
    tl.font = Font(bold=True, size=11)
    tl.alignment = ctr_align
    tl.border = border
    tl.fill = total_fill

    for col, val in zip([4, 5, 6, 7], [tot_brut, tot_irg, tot_net, tot_transf]):
        tc = ws.cell(row=total_row, column=col, value=val)
        tc.font           = Font(bold=True, color="1F4E79", size=10)
        tc.number_format  = '#,##0.00'
        tc.alignment      = amt_align
        tc.fill           = total_fill
        tc.border         = border

    for col in [8, 9, 10, 11, 12]:
        ws.cell(row=total_row, column=col).border = border
        ws.cell(row=total_row, column=col).fill   = total_fill

    ws.freeze_panes = "A5"
    wb.save(output_path)

    print(f"\n✅ Excel sauvegardé : {output_path}")
    print(f"   Total employés         : {len(lignes)}")
    print(f"   Total Salaire Brut     : {tot_brut:>18,.2f}")
    print(f"   Total IRG              : {tot_irg:>18,.2f}")
    print(f"   Total Salaire Net      : {tot_net:>18,.2f}")
    print(f"   Total Montant Transf.  : {tot_transf:>18,.2f}")


export_excel(all_lignes, EXCEL_PATH, PERIODE)

## 7. Vérification finale

In [ ]:
import os
size_kb = os.path.getsize(EXCEL_PATH) / 1024
print(f"✅ Fichier Excel généré")
print(f"   Chemin  : {EXCEL_PATH}")
print(f"   Taille  : {size_kb:.1f} KB")
print(f"\n🎯 Colonnes Excel :")
print(f"   A  — N°")
print(f"   B  — Matricule")
print(f"   C  — Nom & Prénom")
print(f"   D  — Salaire Brut")
print(f"   E  — IRG")
print(f"   F  — Salaire Net")
print(f"   G  — Montant Transférable")
print(f"   H  — Banque (source PDF)")
print(f"   I  — Période")
print(f"   J  — N° Compte  ← issu du fichier banque")
print(f"   K  — Nom Banque (match)")
print(f"   L  — Type Match (EXACT / FUZZY / NON TROUVÉ)")
print(f"\n🎨 Code couleur :")
print(f"   Vert   — match exact")
print(f"   Jaune  — match fuzzy (à valider)")
print(f"   Rouge  — aucun match trouvé")